# 用 fine-tuning 讓 gpt-oss 的韓文表現更好

> 原文：[articles/gpt-oss/fine-tune-korean.ipynb](https://github.com/openai/openai-cookbook/blob/main/articles/gpt-oss/fine-tune-korean.ipynb)



## 這篇在做什麼

這本 notebook 示範怎麼用 LoRA / QLoRA 去 fine-tune OpenAI 的 open-weight 模型 gpt-oss，讓它寫得出韓國新聞文體和現代聊天口吻。流程包含環境檢查、準備資料、去除個資、訓練、合併 adapter 和量化。原文是韓文加英文雙語，這裡統一翻成中文。

## 學完你會

- 知道 gpt-oss 能怎麼訓練：用 BF16 或 QLoRA 4-bit 訓練，訓練完再後處理量化成 MXFP4，不能直接在 MXFP4 上訓練。
- 會設定 LoRA 的 target（只打 attention，或加上 MoE expert 層）和 r、alpha、dropout 等超參數。
- 能走完一條完整流程：做樣本資料、遮蔽個資、貼上風格標籤、訓練、合併 adapter、存檔、評估。

## 開跑前

- 需要 GPU，建議用 Colab 選 GPU runtime，模型以 BF16 為主；CPU 或 Mac 只適合做輕量測試。
- 套件版本（`transformers`、`peft`、`accelerate`、`trl`）要能支援 BF16 / 4-bit LoRA。

In [ ]:
# 環境設定：在 Colab 左側 🔑 Secrets 新增 OPENAI_API_KEY
%pip install -q -U datasets openai_harmony peft torch transformers trl
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # 本機執行就用自己的環境變數

# 抓原 repo 的資料檔（只下載這個資料夾）
if not os.path.exists('openai-cookbook'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/openai/openai-cookbook
    !cd openai-cookbook && git sparse-checkout set articles/gpt-oss
os.chdir('openai-cookbook/articles/gpt-oss')


這本 notebook 示範怎麼 fine-tune OpenAI 的 **gpt-oss（open-weight）** 模型，讓它具備**韓國新聞文體加上現代聊天口吻**。

---

### MXFP4 流程釐清

- 目前公開的框架**都不支援直接在 MXFP4 上訓練或 fine-tuning**。
- 建議路線：用 **BF16**（或 **QLoRA 4-bit nf4**）訓練，接著**合併 LoRA**，再做**訓練後量化成 MXFP4**，最後用 `save_pretrained()` 存成部署用的檔案。
- 如果你需要 MXFP4 的成品，合併 adapter 之後必須**從 BF16 重新量化**。（匯出工具還在演進，如果你的工具鏈已經支援 MXFP4 序列化，那是最理想的。）

---

### LoRA 的 target（含 MoE）

- 最小設定（快、省 VRAM）：只打 attention，例如 `["q_proj","v_proj"]`。
- MoE-aware 設定（領域適應更好，但吃更多 VRAM 和時間）：除了 attention，再加上 **expert projection 層**。

```python
from peft import LoraConfig

TARGET_MODULES = ["q_proj", "v_proj"]  # baseline
MOE_TARGET_PARAMETERS = [
    # example expert layers; adjust indices to your model depth
    "mlp.experts.gate_up_proj",
    "mlp.experts.down_proj",
]

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear",              # cover all linear layers
    target_parameters=MOE_TARGET_PARAMETERS,  # add expert projections
    bias="none", task_type="CAUSAL_LM",
)
```

- 先只打 attention，如果韓文領域的表現不夠好，再打開 MoE target 重新評估。

## 目錄
0) 目標與範圍  
1) 環境檢查  
2) 設定值  
3) 安裝套件  
4) 資料來源（韓國情境）  
5) 產生樣本資料  
6) 前處理（PIPA）與風格標籤  
7) 資料載入與格式化  
8) 載入模型與 tokenizer  
9) Fine-tuning（LoRA/QLoRA）  
   9a) 資料整理與切分  
   9b) 超參數（r/alpha/dropout）  
   9c) 合併 adapter（BF16）  
   9d) 儲存合併後的 BF16（`save_pretrained`）  
   9e) 匯出與量化（BF16 → MXFP4）  
10) 評估（新聞與對話）  
11) 推論用 prompt 範本  
12) 保持時效性  
13) 安全與合規  
14) 疑難排解與下一步  


### ⚙️ 訓練 vs 量化：哪些是支援的
- **可以：** 用 BF16/FP16 或 QLoRA 訓練，匯出合併後的權重。
- **接著：** 用現成的轉換腳本或工具，把權重量化成 **MXFP4** 來做推論。
- **不要：** 想跑端到端「在 MXFP4 上訓練」的流程，目前不支援。

> **個資與合規提醒：** 韓文資料請照公司政策處理（遮蔽身分證字號 RRN、電話、帳號，移除 email），而且要在訓練和寫 log **之前**做。train/val/test 的切分要依資料來源和風格標籤分層。

### 🧪 MoE adapter（選用）
你可以用 adapter 打 MoE 層，不過這屬於**進階、實驗性**的做法。先從 attention projection 開始，驗證完韓文 benchmark 再擴大範圍。

> **注意：** `transformers`、`peft`、`accelerate`、`trl` 的版本要固定在已知支援 BF16 / 4-bit LoRA 的版本。  
如果你有固定 `safetensors` 的版本，要記得**原生 MXFP4 序列化還沒標準化**，載入時可能會在內部升精度。

### 🔎 支援矩陣一覽
- **Fine-tuning 精度：** BF16/FP16 ✅ · QLoRA 4-bit ✅ · **MXFP4 FT ❌**
- **量化目標：** MXFP4 ✅（訓練後）
- **OSS 模型的託管 API FT：** ❌
- **開源 FT（Transformers/TRL/PEFT）：** ✅
- **LoRA targets：** `q_proj`、`k_proj`、`v_proj`、`o_proj` ✅；MoE expert adapter **實驗性** ⚠️

---

## 0) 目標與範圍
- 針對韓文一般新聞加上日常/諮詢對話口吻做最佳化，用 `style=news_headline|news_lead|news_body|kakao_casual|kakao_formal` 控制輸出風格。
- **Stack**：`transformers`、`trl(SFTTrainer)`、`peft(LoRA/QLoRA)`、`datasets`。
- **硬體**：單張或少量 GPU（偏好 BF16）。CPU/Mac 只做輕量測試。

## 1) 環境檢查

In [9]:
import os, sys, platform
print("Python:", sys.version)
print("OS/Platform:", platform.platform())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", ""))

try:
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not installed or GPU not detected:", e)

Python: 3.10.12 (main, May 27 2025, 17:12:29) [GCC 11.4.0]
OS/Platform: Linux-6.8.0-60-generic-x86_64-with-glibc2.35
CUDA_VISIBLE_DEVICES: 
Torch: 2.7.1+cu126 CUDA: True
GPU: NVIDIA H100 80GB HBM3


## 2) 設定值

In [10]:
from pathlib import Path
import os

# === Model & Training Params ===
BASE_URL = "http://localhost:8000/v1"     # vLLM OpenAI-compatible endpoint
API_KEY  = "dummy-key"                     # vLLM ignores; SDK requires a value
MODEL    = "openai/gpt-oss-120b"           # must match the model vLLM loaded
OUTPUT_DIR = "ft-oss-kr-news-chat-bilingual"

# Data mix (news : chat)
MIX_NEWS = 0.6
MIX_CHAT = 0.4

# LoRA
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]  # adjust per model

# Training
EPOCHS = 1
PER_DEVICE_BS = 2
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
BF16 = True
LOG_STEPS = 20
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 2

print("Config ready.")

Config ready.


## 3) 安裝套件

In [11]:
# %pip install --upgrade pip
# %pip install transformers accelerate datasets peft trl bitsandbytes sentencepiece
# (optional) serving/runtimes
# %pip install vllm
# %pip install llama-cpp-python

import importlib, pip

for dep in ["transformers","accelerate","datasets","peft","trl",
            "bitsandbytes","sentencepiece","vllm","llama_cpp"]:
    try:
        print(f"{dep}: {importlib.import_module(dep).__version__}")
    except Exception:
        print(f"{dep}: not installed")

print(f"pip: {pip.__version__}")

print("Install cells are commented. Un-comment in your environment.")

transformers: 4.55.3
accelerate: 1.10.0
datasets: 4.0.0
peft: not installed
trl: 0.21.0
bitsandbytes: not installed
sentencepiece: 0.2.1
vllm: 0.10.1
llama_cpp: 0.3.16
pip: 25.2
Install cells are commented. Un-comment in your environment.


## 4) 資料來源（韓國情境）

- 以公開 benchmark（主題分類、摘要、QA）加上**許可的新聞 API metadata（標題/摘要/版面）**為主，用來校正風格。
- 拿文章**全文大量重新訓練會有著作權和服務條款問題**，所以以 metadata 和公開語料為主。
- 對話體優先用合法的公開語料（有標註半語/敬語、表情符號、縮寫）。
- PIPA（韓國個資法）：身分證字號、電話、email、帳號等個資，要在**訓練前、寫 log 前**先清掉。

## 5) 產生樣本資料

In [12]:
import json, pathlib
pathlib.Path("data").mkdir(exist_ok=True)

news_samples = [
  {"style":"news_lead","topic":"경제","title":"반도체 수출 호조… 7월 수출액 20% 증가","summary":"수출 개선세가 이어지며 경기 회복 기대가 커졌다."},
  {"style":"news_headline","topic":"정치","title":"국회, 데이터 산업 육성법 본회의 통과","summary":"데이터 활용 촉진과 개인정보 보호를 강화하는 내용."},
  {
    "style": "news_lead",
    "topic": "경제",
    "title": "카카오페이 보안 점검… 고객문의: help+vip@corp.co.kr",
    "summary": "고객센터 010-1234-5678로 문의 폭주. 계좌 110-123-456789 관련 결제 오류 논란."
  },
  {
    "style": "news_headline",
    "topic": "사회",
    "title": "개인정보 유출 의혹… 주민번호 901010-1234567 유통 주장",
    "summary": "서울특별시 강남구 테헤란로 123에서 자료 확보… 담당자 john.doe+news@example.com"
  }
]

chat_samples = [
  {"style":"kakao_casual","dialog":["주말에 비 온대?","응 일요일에 꽤 온다더라 ☔","헐 우산 챙겨야겠다"]},
  {"style":"kakao_formal","dialog":["안녕하세요. 배송 일정 확인 부탁드립니다.","내일 중 도착 예정입니다.","안내 감사합니다."]},
  {
    "style": "kakao_formal",
    "dialog": [
      "배송 확인 부탁드립니다. 주문번호 ORD-2025-0001 입니다.",
      "연락처는 010-2222-3333 입니다. (유니코드 하이픈)",
      "주민등록번호는 제공할 수 없습니다."
    ]
  }
]

with open("data/news.jsonl","w",encoding="utf-8") as f:
    for ex in news_samples: f.write(json.dumps(ex, ensure_ascii=False)+"\n")
with open("data/chat.jsonl","w",encoding="utf-8") as f:
    for ex in chat_samples: f.write(json.dumps(ex, ensure_ascii=False)+"\n")

print("Created: data/news.jsonl, data/chat.jsonl")

Created: data/news.jsonl, data/chat.jsonl


## 6) 前處理（PIPA）與風格標籤

In [13]:
# Step 6 — PII scrubbing + style tags (no Harmony here)
import json, re, unicodedata
from pathlib import Path

# --- Normalization helpers ---
HYPHENS = dict.fromkeys(map(ord, "‐-‒–—―﹘﹣－"), ord("-"))  # map unicode hyphens → ASCII
def normalize(s: str) -> str:
    if not isinstance(s, str): return s
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(HYPHENS)
    return s

# --- PII patterns (illustrative; tune for production) ---
RE_EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
# KR mobile numbers with spaces/hyphens: 010-1234-5678, 010 1234 5678, etc.
RE_PHONE = re.compile(r"\b01[016789][-\s]?\d{3,4}[-\s]?\d{4}\b")
# Korean RRN (주민등록번호) basic pattern
RE_RRN = re.compile(r"\b\d{6}-\d{7}\b")
# Bank-ish account numbers: strictly digits in groups (avoid codes with letters)
RE_ACCOUNT = re.compile(r"\b\d{2,3}-\d{2,4}-\d{3,6}\b")
# Very simple postal address cue (city names) – conservative, just redact the token (optional)
RE_CITY = re.compile(r"(서울특별시|부산광역시|대구광역시|인천광역시|광주광역시|대전광역시|울산광역시|세종특별자치시|경기도|강원도|충청북도|충청남도|전라북도|전라남도|경상북도|경상남도|제주특별자치도)")

# Allowlist: things that look like PII but aren’t (e.g., bill/order codes w/ letters)
def looks_like_code(s: str) -> bool:
    return bool(re.search(r"[A-Za-z]", s))  # if letters present, treat as code, not account/phone

# Order of application matters (longest/most specific first sometimes helps)
SCRUBBERS = [
    ("[RRN]", RE_RRN),
    ("[EMAIL]", RE_EMAIL),
    ("[PHONE]", RE_PHONE),
    ("[ACCOUNT]", RE_ACCOUNT),
    ("[CITY]", RE_CITY),  # optional; comment out if you don't want to redact city tokens
]

def scrub_text(text: str) -> tuple[str, dict]:
    """Return (scrubbed_text, hits_dict). Avoid false positives with basic allowlisting."""
    if not isinstance(text, str) or not text:
        return text, {}
    orig = text
    text = normalize(text)
    hits = {}

    # Guard account-like and phone-like strings that contain letters (likely codes)
    guarded = set()
    for m in RE_ACCOUNT.finditer(text):
        if looks_like_code(m.group(0)):
            guarded.add(m.span())
    for m in RE_PHONE.finditer(text):
        if looks_like_code(m.group(0)):
            guarded.add(m.span())

    # Apply scrubs
    for label, pattern in SCRUBBERS:
        out = []
        last = 0
        count = 0
        for m in pattern.finditer(text):
            span = m.span()
            if pattern in (RE_ACCOUNT, RE_PHONE) and span in guarded:
                continue
            out.append(text[last:span[0]])
            out.append(label)
            last = span[1]
            count += 1
        out.append(text[last:])
        text = "".join(out)
        if count:
            hits[label] = hits.get(label, 0) + count

    return text, hits if text != orig else {}

def scrub_record(rec: dict, kind: str) -> tuple[dict, dict]:
    """Scrub fields in a news/chat record; return (new_rec, hits)."""
    rec = dict(rec)  # shallow copy
    total_hits = {}

    def scrub_field(key):
        val = rec.get(key)
        new, hits = scrub_text(val) if isinstance(val, str) else (val, {})
        rec[key] = new
        for k, v in hits.items():
            total_hits[k] = total_hits.get(k, 0) + v

    if kind == "news":
        for key in ("title", "summary", "topic"):
            scrub_field(key)
    elif kind == "chat":
        scrub_field("style")
        if isinstance(rec.get("dialog"), list):
            cleaned_dialog = []
            for turn in rec["dialog"]:
                new, hits = scrub_text(turn) if isinstance(turn, str) else (turn, {})
                cleaned_dialog.append(new)
                for k, v in hits.items():
                    total_hits[k] = total_hits.get(k, 0) + v
            rec["dialog"] = cleaned_dialog

    return rec, total_hits

# --- Style tagger (lightweight labels for later routing/metrics) ---
def build_style_tags(rec: dict, kind: str) -> list[str]:
    tags = []
    if kind == "news":
        tags.append("domain:" + (rec.get("topic") or "unknown"))
        tags.append("style:" + (rec.get("style") or "news"))
        tags.append("tone:formal")
        tags.append("medium:news")
    elif kind == "chat":
        style = (rec.get("style") or "").lower()
        tags.append("style:" + (style or "chat"))
        tags.append("tone:" + ("formal" if "formal" in style else "casual"))
        tags.append("medium:kakao")
    return [t.replace(" ", "_") for t in tags]

# --- Process files ---
def process_file(src: str, dst: str, kind: str):
    total = 0
    redacted = 0
    counters = {}
    with open(src, encoding="utf-8") as fin, open(dst, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip(): continue
            rec = json.loads(line)
            total += 1
            cleaned, hits = scrub_record(rec, kind)
            cleaned["style_tags"] = build_style_tags(cleaned, kind)
            cleaned["_pii_hits"] = hits  # keep for inspection; drop later if you want
            if hits: redacted += 1
            for k, v in hits.items():
                counters[k] = counters.get(k, 0) + v
            fout.write(json.dumps(cleaned, ensure_ascii=False) + "\n")
    print(f"{src} -> {dst} | rows: {total}, redacted_rows: {redacted}, hits: {counters}")

process_file("data/news.jsonl", "data/news_clean.jsonl", kind="news")
process_file("data/chat.jsonl", "data/chat_clean.jsonl", kind="chat")

data/news.jsonl -> data/news_clean.jsonl | rows: 4, redacted_rows: 2, hits: {'[EMAIL]': 2, '[ACCOUNT]': 1, '[RRN]': 1, '[CITY]': 1}
data/chat.jsonl -> data/chat_clean.jsonl | rows: 3, redacted_rows: 1, hits: {'[PHONE]': 1}


## 7) 資料載入與格式化

In [15]:
# Step 7 — Harmony conversion + dataset loading & tokenization
import json, math
from pathlib import Path
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import AutoTokenizer

DATA = Path("data")
assert (DATA / "news_clean.jsonl").exists(), "Run Step 6 first"
assert (DATA / "chat_clean.jsonl").exists(), "Run Step 6 first"

# ---------- 7A) Convert cleaned → Harmony messages ----------

def news_to_messages(rec):
    # system style from Step 6 tags; default to KR news tone
    system = "한국 뉴스 문체로 간결하고 사실 위주로 작성."
    # user asks for a headline+lead from topic; assistant is the expected formatted answer
    user = f"주제: {rec.get('topic','알수없음')}. 기사 제목과 요약을 생성해줘."
    assistant = f"{rec.get('title','')} — {rec.get('summary','')}"
    return [{"role":"system","content":system},
            {"role":"user","content":user},
            {"role":"assistant","content":assistant}]

def chat_to_messages(rec):
    # Keep style hint (casual/formal) in system
    style = (rec.get("style") or "").lower()
    system = f"카카오톡 대화 스타일. style={style or 'chat'}"
    dialog = rec.get("dialog") or []
    msgs = [{"role":"system","content":system}]
    # Alternate user/assistant turns; if odd length, last user stays without assistant label
    roles = ["user","assistant"]
    for i, turn in enumerate(dialog[:6]):  # cap tiny demos to avoid runaway
        msgs.append({"role": roles[i % 2], "content": str(turn)})
    # Ensure there is at least one assistant turn for SFT
    if not any(m["role"]=="assistant" for m in msgs):
        msgs.append({"role":"assistant","content":"네, 확인했습니다."})
    return msgs

def write_harmony(src, dst, kind):
    convert = news_to_messages if kind=="news" else chat_to_messages
    with open(src, encoding="utf-8") as fin, open(dst, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip(): continue
            rec = json.loads(line)
            msgs = convert(rec)
            fout.write(json.dumps({"messages": msgs}, ensure_ascii=False) + "\n")

write_harmony(DATA/"news_clean.jsonl", DATA/"news_harmony.jsonl", "news")
write_harmony(DATA/"chat_clean.jsonl", DATA/"chat_harmony.jsonl", "chat")
print("Created:", DATA/"news_harmony.jsonl", DATA/"chat_harmony.jsonl")

# ---------- 7B) Load Harmony JSONL with 🤗 Datasets ----------
raw = load_dataset(
    "json",
    data_files={"news": str(DATA/"news_harmony.jsonl"),
                "chat": str(DATA/"chat_harmony.jsonl")}
)

# Mix train split using your Step-2 mix ratios
news = raw["news"]
chat = raw["chat"]

def take_portion(ds, frac):
    n = max(1, int(round(len(ds) * frac)))
    return ds.select(range(n)) if n < len(ds) else ds

news_part = take_portion(news, MIX_NEWS if 'MIX_NEWS' in globals() else 0.5)
chat_part = take_portion(chat, MIX_CHAT if 'MIX_CHAT' in globals() else 0.5)
train_ds = concatenate_datasets([news_part, chat_part]).shuffle(seed=42)

# Tiny validation built from remaining examples (if any)
remaining_news = news.select(range(len(news_part), len(news))) if len(news) > len(news_part) else news_part
remaining_chat = chat.select(range(len(chat_part), len(chat))) if len(chat) > len(chat_part) else chat_part
val_candidates = concatenate_datasets([remaining_news, remaining_chat])
val_ds = val_candidates.shuffle(seed=43).select(range(min(64, len(val_candidates)))) if len(val_candidates) else train_ds.select(range(min(32, len(train_ds))))

dataset = {"train": train_ds, "validation": val_ds}
print({k: len(v) for k, v in dataset.items()})


Created: data/news_harmony.jsonl data/chat_harmony.jsonl


Generating news split: 0 examples [00:00, ? examples/s]

Generating chat split: 0 examples [00:00, ? examples/s]

{'train': 3, 'validation': 4}


## 8) 載入模型與 tokenizer

In [16]:
# ---------- 7C) Tokenizer + Harmony template fallback ----------
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    use_fast=True,          # required if only tokenizer.json exists
    trust_remote_code=True,
    force_download=True     # ensures a fresh pull
)

if not getattr(tokenizer, "chat_template", None):
    # Minimal Harmony-style fallback (server already knows Harmony; this is ONLY for training tokenization)
    tokenizer.chat_template = """{% for m in messages -%}
{%- if m['role'] == 'system' -%}<|system|>
{{ m['content'] }}<|end|>
{%- elif m['role'] == 'user' -%}<|user|>
{{ m['content'] }}<|end|>
{%- elif m['role'] == 'assistant' -%}<|assistant|>
{{ m['content'] }}<|end|>
{%- endif -%}
{%- endfor -%}"""

# Ensure pad/eos are sane
tokenizer.pad_token = tokenizer.eos_token or tokenizer.pad_token

# ---------- 7D) Tokenize with assistant-only labels ----------
ASST_TOKEN = None
END_TOKEN = None
try:
    ASST_TOKEN = tokenizer.convert_tokens_to_ids("<|assistant|>")
    END_TOKEN = tokenizer.convert_tokens_to_ids("<|end|>")
except Exception:
    # If the base vocab lacks these tokens, it's okay; masking fallback below will still work heuristically
    pass

MAX_LEN = 2048  # you can raise this if you have room

def tokenize_with_labels(example):
    # 1) Render with chat template (includes assistant answer)
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    # 2) Tokenize
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN)
    input_ids = enc["input_ids"]
    labels = [-100] * len(input_ids)

    # 3) Label only assistant content
    if ASST_TOKEN is not None and END_TOKEN is not None:
        start = None
        for i, tid in enumerate(input_ids):
            if tid == ASST_TOKEN:
                start = i + 1  # learn after the tag
            elif start is not None and tid == END_TOKEN:
                start = None
            elif start is not None:
                labels[i] = input_ids[i]
    else:
        # Heuristic fallback: learn on the last third of tokens (crude but avoids total silence)
        start = int(len(input_ids) * 0.66)
        for i in range(start, len(input_ids)):
            labels[i] = input_ids[i]

    return {"input_ids": input_ids, "attention_mask": enc["attention_mask"], "labels": labels}

tokenized_train = dataset["train"].map(tokenize_with_labels, remove_columns=["messages"])
tokenized_val   = dataset["validation"].map(tokenize_with_labels, remove_columns=["messages"])

print("Tokenization done.",
      "train:", len(tokenized_train),
      "val:", len(tokenized_val),
      "example lens:", tokenized_train[0]["input_ids"][:12], "...")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenization done. train: 3 val: 4 example lens: [200006, 17360, 200008, 3575, 553, 17554, 162016, 11, 261, 4410, 6439, 2359] ...


## 9) Fine-tuning（LoRA/QLoRA）
### 9a) 資料整理與切分
_（資料集準備請看第 7、8 節，需要的話把相關程式碼搬到這裡。）_
### 9b) 超參數（r/alpha/dropout）
```python
# Example LoRA hyperparameters
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
```

### 9c) 合併 adapter（BF16）
```python
# Example merge step (after training)
# model = PeftModel.from_pretrained(base_model, adapter_path)
# merged_model = model.merge_and_unload()
```

### 9d) 儲存合併後的 BF16（`save_pretrained`）
```python
# merged_model.save_pretrained(OUTPUT_DIR)
```


### 9e) 匯出與量化（BF16 → MXFP4）

**中立、不綁特定框架的做法：**  
目前公開的函式庫**不支援**直接在 MXFP4 上訓練或 fine-tuning。常見的流程是：
1) 用 **BF16**（或 **QLoRA 4-bit nf4**）做**訓練/SFT**。  
2) 把 **LoRA adapter 合併**進 base model（BF16）。  
3) 用 `save_pretrained()` **儲存**合併後的 BF16 checkpoint。  
4) 用**廠商或工具鏈提供的 packer**，把合併後的 BF16 tensor 做**訓練後量化成 MXFP4**。  
5) **儲存/匯出** MXFP4 成品（結構跟 Hugging Face `save_pretrained()` 的輸出一樣），拿去部署或 serving。

> 備註：  
> - 如果你的 serving 環境支援**推論時載入 LoRA**，可以省略合併和量化，直接交付：**base（MXFP4 或 BF16）加 LoRA adapter**。  
> - 如果 runtime 要的是**合併後的 MXFP4**，合併 adapter 之後就必須跑一次 **BF16 → MXFP4** 量化。  
> - BF16 和 MXFP4 兩份匯出的 **tokenizer/config** 檔要保持一致。


In [19]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    task_type="CAUSAL_LM",
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES
)

# base_model = get_peft_model(base_model, lora_cfg)

sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    bf16=BF16,
    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT
)

# trainer = SFTTrainer(model=base_model, args=sft_args, train_dataset=combined, tokenizer=tokenizer)
# trainer.train()
# trainer.save_model(OUTPUT_DIR)
print("Fine‑tuning skeleton ready. Un‑comment on your machine.")

Fine‑tuning skeleton ready. Un‑comment on your machine.


## 10) 評估（新聞與對話）

**韓文指標**  
- 新聞性：主題分類吻合度（F1）、摘要品質（ROUGE-1/2/L）、閱讀理解 QA（EM/F1）。  
- 對話性：自然度與上下文維持、敬語與半語切換的正確率、表情符號與縮寫用得合不合適。

**補充**  
- 在授權允許的前提下，使用韓文公開 benchmark（例如主題分類、類似 KorQuAD 的 QA）。
- 自動指標（F1/ROUGE）要搭配人工評估，才看得出語氣和禮貌程度。

In [20]:
# Example helpers (stub)
def simple_accuracy(preds, labels):
    return sum(int(p==g) for p,g in zip(preds, labels)) / max(1, len(labels))

# For ROUGE:
# import evaluate
# rouge = evaluate.load("rouge")
# result = rouge.compute(predictions=pred_texts, references=ref_texts)
# print(result)

print("Eval stubs ready.")

Eval stubs ready.


## 11) 推論用 prompt 範本

In [25]:
from openai_harmony import Message, ChatFormatter

# Example prompt construction using Harmony
messages = [
    Message(role="system", content="너는 한국 고객을 돕는 유능한 AI 어시스턴트다."),
    Message(role="user", content="국내 PIPA 규정을 준수하면서 사내 문서 요약기를 구성하려면 어떤 아키텍처가 좋을까?")
]

prompt = ChatFormatter.to_chat_prompt(messages)
print(prompt)  # For preview; pass to tokenizer when running inference


<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-21

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

너는 한국 고객을 돕는 유능한 AI 어시스턴트다.

<|end|><|start|>user<|message|>국내 PIPA 규정을 준수하면서 사내 문서 요약기를 구성하려면 어떤 아키텍처가 좋을까?<|end|><|start|>assistant


## 12) 保持時效性

- **每週校正 SFT**：抽樣許可的新聞 API **metadata（標題/摘要/版面）**，用來校正風格。  
- **更新對話體**：把最新的縮寫、新造詞、表情符號詞典加進去（例如 ㄱㄱ、ㅇㅋ、ㅋㅋ、ㄹㅇ）。  
- **回歸評估**：用同樣的指標比較前後差異，再調整混合比例、temperature 和 penalty。

## 13) 安全與合規

- 確認資料來源和授權（benchmark、API、內部資料）。
- 在訓練、寫 log、評估之前先清除個資。
- 遵守著作權和服務條款（**禁止拿文章全文大量重新訓練**）。
- 驗證輸出（schema、禁用詞、敏感度規則）。
- 管理資料集、模型的版本，並保留評估報告。

## 14) 疑難排解與下一步

- 調整混合比例：（新聞:對話）6:4 改成 7:3 或 5:5  
- LoRA 超參數：r=8~16、α=16~32、dropout=0.05~0.1  
- 上線服務：用 vLLM/llama.cpp serving，再加上主題/風格的路由  
- 搭配 RAG：接上新聞或文件索引，補強最新事實的正確性  
- A/B 測試：衡量語氣、長度、表情符號用量等使用者滿意度

## 重點整理

- gpt-oss 目前沒辦法直接在 MXFP4 上訓練，要先用 BF16 或 QLoRA 訓練，合併 adapter 後再量化。
- LoRA 先只打 attention，不夠再加 MoE expert 層，這樣比較省資源也好驗證。
- 個資清除和授權確認要放在訓練與 logging 之前，而且拿新聞全文大量訓練有法律風險。
- 風格標籤（`style=...`）讓同一個模型能切換新聞體和聊天體。
- 模型上線後要靠每週校正和回歸評估，維持用語的時效性。

## 練習

1. 把 9b 的 `LORA_R` 從 8 改成 16、`LORA_ALPHA` 改成 32，想想這對可訓練參數量和 VRAM 有什麼影響。
2. 在最上面 `LoraConfig` 的例子裡，把 `target_modules` 從 `"all-linear"` 改成 `TARGET_MODULES`（只打 `q_proj`、`v_proj`），再決定要不要保留 `target_parameters`，並說明兩種設定各適合什麼情況。
3. 在第 5、6 節的樣本資料和去個資程式碼裡，新增一種你自己的風格標籤或一個新的個資格式（例如統一編號），再用評估那段程式確認有被處理到。